In [1]:
!pip show azure-ai-ml

Name: azure-ai-ml
Version: 1.31.0
Summary: Microsoft Azure Machine Learning Client Library for Python
Home-page: https://github.com/Azure/azure-sdk-for-python
Author: Microsoft Corporation
Author-email: azuresdkengsysadmins@microsoft.com
License: MIT License
Location: C:\Users\dgouwy\Documents\devoProjects\azure-machine-learning\.venv\Lib\site-packages
Requires: azure-common, azure-core, azure-mgmt-core, azure-monitor-opentelemetry, azure-storage-blob, azure-storage-file-datalake, azure-storage-file-share, colorama, isodate, jsonschema, marshmallow, pydash, pyjwt, pyyaml, strictyaml, tqdm, typing-extensions
Required-by: 


## Connect to the workspace

**Default Azure Credentials vs. Interactive Browser Credentials**

Default Azure Credentials (from Gemini):
> Think of this as the "Smart Auto-Login." It is a chained credential that looks for authentication in a specific order. If it finds one  that works, it stops searching and uses it.
>   - It typically checks in this order:
>   - Environment Variables: Looks for service principal secrets (AZURE_CLIENT_ID, etc.).
>   - Workload/Managed Identity: Used when your code is running inside Azure (like on an Azure ML Compute Instance or a Function App).
>   - Azure CLI / Azure PowerShell: It steals the "session" from your terminal if you’ve already run az login.
>   - Visual Studio Code: Uses the account signed into the Azure Account extension.
>

Interactive Browser Credentials (from Gemini):
> This is the "Manual Login." When this code runs, it will:
>
>   - Open your default system web browser.
>   - Direct you to the Microsoft login page.
>   - Wait for you to enter your username, password, and MFA (Multi-Factor Authentication).

In [2]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient
import os

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default") # Chech if we can get a token
except Exception as e:
    InteractiveBrowserCredential()

In [3]:
ml_client = MLClient(
     credential             = credential
    ,subscription_id        = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name    = os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name         = os.environ["AZURE_ML"]
    )

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## Diabetes training-script

In [6]:
%%writefile ./code/diabetes-training.py

import pandas as pd
import numpy as np
import logging
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.preprocessing import MinMaxScaler

# 1. Get the data

df = pd.read_csv("../../data/diabetes-data/diabetes.csv")

# 2. Preprocess

X, y = df.drop(["PatientID","Diabetic"], axis=1).values, df["Diabetic"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train model

reg = 0.01
model = LogisticRegression(C=1/reg, solver="liblinear").fit(X_train_scaled, y_train)
preds = model.predict(X_test_scaled)

# 4. Evaluate

acc = np.average(preds == y_test)
print(f"Accuracy: {acc}")

pred_scores = model.predict_proba(X_test)
auc = roc_auc_score(y_test,pred_scores[:,1])
print('AUC: ' + str(auc))

Overwriting ./code/diabetes-training.py


In [7]:
# Test if it works
with open("./code/diabetes-training.py", "r") as f:
    exec(f.read())

Accuracy: 0.774
AUC: 0.5


## Upload as component

(And try a little pipeline)

In [1]:
from azure.ai.ml import command
from azure.ai.ml.entities import AmlCompute
import time

### 1. Create compute

compute_name = "diabetes-cluster"

aml_compute = AmlCompute(
     name                           = compute_name
    ,size                           = "Standard_DS11_v2"
    ,min_instances                  = 0
    ,max_instances                  = 4
    ,idle_time_before_scale_down    = 30
)

try:
    poller = ml_client.begin_create_or_update(aml_compute)
    while not poller.done:
        print(f"Setting up compute: {compute_name}.", end="\r")
    print(f"Compute {compute_name} created")
except Exception as e:
    raise e

### 2. Create job
job = command(
     code               = "./code"
    ,command            = "python diabetes-training.py"
    ,environment        = "AzureML-sklearn-1.5@latest"
    ,compute            = compute_name
    ,display_name       = "diabetes-train-component"
    ,experiment_name    = "diabetes-training"  
)

while True:
    time.sleep(1)
    compute_state = ml_client.compute.get(compute_name).provisioning_state.lower()
    if compute_state == "succeeded":
        returned_job = ml_client.create_or_update(job)
        break 
print(returned_job.studio_url)


### 3. Cleanup

while True:
    time.sleep(2)
    job_state = ml_client.jobs.get(returned_job.name).status.lower()
    if job_state in ["completed", "failed", "canceled"]:
        try:
            poller = ml_client.compute.begin_delete(compute_name)
            while not poller.done:
                print(f"Deleting compute: {compute_name}.", end="\r")
            print({f"Compute {compute_name} deleted"})
        except Exception as e:
            raise e
        break


NameError: name 'ml_client' is not defined

In [56]:
ml_client.jobs.get(returned_job.name).status

'Failed'

'Succeeded'

In [36]:
job = command(
     code               = "./code"
    ,command            = "python diabetes-training.py"
    ,environment        = "AzureML-sklearn-1.5@latest"
    ,compute            = compute_name
    ,display_name       = "diabetes-train-component"
    ,experiment_name    = "diabetes-training"  
)

returned_job = ml_client.create_or_update(
    job
)
print(returned_job.studio_url)

https://ml.azure.com/runs/gentle_pencil_1h43zk3p9x?wsid=/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science&tid=d018aec4-2b2b-4c66-9939-2c96877e6bf1
